# Advanced Solar Power Forecasting
This notebook trains and evaluates a hybrid ensemble model combining **LightGBM** and **XGBoost**, optimized through both Grid Search and Bayesian Search, for predicting solar power generation. Note: only a part of the code is available publicly.
The goal is to achieve high accuracy using only weather and location-based features — without solar irradiance data.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from pysolar.solar import get_altitude, get_azimuth
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import dask.dataframe as dd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
pip install dask
pip install pysolar
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.feature_selection import RFE
pip install shap
import shap
import pandas as pd
import numpy as np
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from skopt import BayesSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import numpy as np
import pickle
from sklearn.feature_selection import RFE
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import gc 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.feature_selection import RFE
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression



In [ ]:
# Load the dataset
file_path = r'C:\Users\mahdi\OneDrive - Toronto Metropolitan University\Desktop\Paper\Materials\Final Project\SolarPower_Prediction\Pasion et al dataset.csv'
df = pd.read_csv(file_path)
print(df.head())
# Step 1: Convert 'Date' from int to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d')
# Step 2: Extract and transform date-related features
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['MonthOfYear'] = df['Date'].dt.month
df['MonthOfYear'] = df['MonthOfYear'].astype(float)
df['DayOfWeek'] = df['DayOfWeek'].astype(float)
df['Month_Day_Interaction'] = df['MonthOfYear'] * df['DayOfWeek']
df.rename(columns={'Month_DayOfWeek': 'Month_Day_Categorical'}, inplace=True)
df.set_index('Date', inplace=True)
df['Time'] = pd.to_numeric(df['Time'], errors='coerce')
# Step 4: Parse 'YRMODAHRMI' column if it exists
if 'YRMODAHRMI' in df.columns:
    def parse_ymodahm(column):
        try:
            dt = pd.to_datetime(column, format='%Y%m%d%H%M')
            return dt.year, dt.month, dt.day, dt.hour, dt.minute
        except ValueError as e:
            print(f"Error parsing {column}: {e}")
            return pd.NaT.year, pd.NaT.month, pd.NaT.day, pd.NaT.hour, pd.NaT.minute

    results = df['YRMODAHRMI'].apply(parse_ymodahm)
    df['Year'], df['Month'], df['Day'], df['Hour'], df['Minute'] = zip(*results)
    df = df.drop(columns=['YRMODAHRMI'])

In [ ]:
# Step 5: Create interactions and cyclic features
df['Month_DayOfWeek'] = df['MonthOfYear'].astype(str) + '_' + df['DayOfWeek'].astype(str)
df['Month_sin'] = np.sin(2 * np.pi * df['MonthOfYear'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['MonthOfYear'] / 12)
df['DayOfWeek_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['DayOfWeek_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
df['AmbientTemp_MA7'] = df['AmbientTemp'].rolling(window=7).mean()
df['WindSpeed_EMA7'] = df['Wind.Speed'].ewm(span=7, adjust=False).mean()
df['Month_Day_Interaction'] = df['MonthOfYear'] * df['DayOfWeek']
df['PartOfDay'] = pd.cut(df['Hour'], bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)
df['Time_squared'] = df['Time'] ** 2
df['Hour_sin'] = np.sin(2 * np.pi * df['Time'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Time'] / 24)
df['Temp_Humidity_Interaction'] = df['AmbientTemp'] * df['Humidity']
df['Temp_Rolling_Mean'] = df['AmbientTemp'].rolling(window=7).mean()
df['Pressure_Rolling_Mean'] = df['Pressure'].rolling(window=7).mean()

# Step 6: Normalize numerical columns
scaler = MinMaxScaler()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])




In [ ]:
# Step 7: Function to calculate solar positions
def calculate_solar_positions(df):
    df['timestamp'] = pd.to_datetime(df.index, utc=True)
    df['Solar_Elevation'] = df.apply(lambda row: get_altitude(row['Latitude'], row['Longitude'], row['timestamp']), axis=1)
    df['Solar_Azimuth'] = df.apply(lambda row: get_azimuth(row['Latitude'], row['Longitude'], row['timestamp']), axis=1)
    return df

# Step 8: Calculate solar positions using Dask
ddf = dd.from_pandas(df, npartitions=10)
meta = df.copy()
meta['timestamp'] = pd.to_datetime(df.index, utc=True)
meta['Solar_Elevation'] = pd.Series(dtype='float')
meta['Solar_Azimuth'] = pd.Series(dtype='float')

if 'Date' in ddf.columns:
    ddf['Date'] = dd.to_datetime(ddf['Date'])
    ddf = ddf.set_index('Date', sorted=True)
else:
    ddf.index = dd.to_datetime(ddf.index)

solar_ddf = ddf.map_partitions(calculate_solar_positions, meta=meta)
df_result = solar_ddf.compute()
df = df_result
# Ensure the necessary columns are present
print("Columns after calculating solar positions:", df.columns)


In [ ]:
# Step 9: Exploring Interaction Terms
if 'AmbientTemp' in df.columns and 'Humidity' in df.columns:
    df['Temp_Humidity_Interaction'] = df['AmbientTemp'] * df['Humidity']
else:
    print("Missing columns for Temp_Humidity_Interaction")

if 'Solar_Elevation' in df.columns and 'Cloud.Ceiling' in df.columns:
    df['Solar_Cloud_Interaction'] = df['Solar_Elevation'] * df['Cloud.Ceiling']
else:
    print("Missing columns for Solar_Cloud_Interaction")

In [ ]:
# Function to clean column names by replacing special characters
def clean_column_names(df):
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_', regex=True)
    return df

In [ ]:
# Step 10: Prepare for model training
X = df_result.drop(['PolyPwr'], axis=1)  # Drop the target variable 'PolyPwr'
y = df_result['PolyPwr']
# Step 11: Convert categorical columns to dummy variables
categorical_cols = ['Location', 'Season', 'Month_DayOfWeek', 'PartOfDay']  # Adjust if needed
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
# Check if the datetime column exists and extract datetime components
if 'Date' in X.columns:
    X['year'] = X['Date'].dt.year
    X['month'] = X['Date'].dt.month
    X['day'] = X['Date'].dt.day
    X['hour'] = X['Date'].dt.hour
    X['minute'] = X['Date'].dt.minute
    X['second'] = X['Date'].dt.second
    X.drop(columns=['Date'], inplace=True)
    # Clean column names to remove special characters
X = clean_column_names(X)


In [ ]:
# Step 11: Drop or convert datetime columns if they exist
if 'timestamp' in X.columns:
    X.drop(columns=['timestamp'], inplace=True)

# step12: Drop any remaining datetime columns if they exist
datetime_cols = X.select_dtypes(include=['datetime', 'datetime64']).columns
X.drop(columns=datetime_cols, inplace=True)

In [ ]:
# Step 13: Identify numerical and categorical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns
# Step 13.1: Identify numerical and categorical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns
# Step 13.2: Define preprocessing for numerical and categorical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
# Step 13.3: Create a column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Step 13.4: Apply preprocessing
X_preprocessed = preprocessor.fit_transform(X)

# Step 13.5: Generate polynomial features
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly = poly.fit_transform(X_preprocessed)

# Convert back to DataFrame to keep track of feature names
feature_names = poly.get_feature_names_out(preprocessor.get_feature_names_out())
X_poly_df = pd.DataFrame(X_poly, columns=feature_names)
print("Preprocessing complete. Data ready for model training.")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_poly_df, y, test_size=0.2, random_state=42)
# Initialize models with optimal parameters
lgbm_model = LGBMRegressor(
    n_estimators=250,
    learning_rate=0.1,
    num_leaves=55,
    reg_alpha=1,
    reg_lambda=0.05,
    random_state=42,
    force_col_wise=True  # Properly formatted
)

xgb_model = XGBRegressor(
    n_estimators=250,
    learning_rate=0.1,
    alpha=1.5,
    reg_lambda=0.015,
    random_state=42
)

# Setup RFECV with KFold for regression
rfecv_lgbm = RFECV(estimator=lgbm_model, step=1, cv=KFold(5), scoring='neg_mean_squared_error')
rfecv_xgb = RFECV(estimator=xgb_model, step=1, cv=KFold(5), scoring='neg_mean_squared_error')

# Fit RFECV
rfecv_lgbm.fit(X_train, y_train)
rfecv_xgb.fit(X_train, y_train)

# Plotting results
plt.figure(figsize=(14, 7))
# LightGBM
plt.subplot(1, 2, 1)
plt.title("RFECV - LightGBM")
plt.xlabel("Number of features selected")
plt.ylabel("Cross-validation score (Neg MSE)")
plt.plot(range(1, len(rfecv_lgbm.cv_results_['mean_test_score']) + 1), rfecv_lgbm.cv_results_['mean_test_score'])
plt.grid(True)

# XGBoost
plt.subplot(1, 2, 2)
plt.title("RFECV - XGBoost")
plt.xlabel("Number of features selected")
plt.ylabel("Cross-validation score (Neg MSE)")
plt.plot(range(1, len(rfecv_xgb.cv_results_['mean_test_score']) + 1), rfecv_xgb.cv_results_['mean_test_score'])
plt.grid(True)

plt.tight_layout()
plt.show()

# Outputting the optimal number of features
optimal_features_lgbm = rfecv_lgbm.n_features_
optimal_features_xgb = rfecv_xgb.n_features_

print("Optimal number of features for LightGBM:", optimal_features_lgbm)
print("Optimal number of features for XGBoost:", optimal_features_xgb)


In [ ]:
# Initialize models with optimal parameters
lgbm_model = LGBMRegressor(
    n_estimators=250,
    learning_rate=0.1,
    num_leaves=55,
    reg_alpha=1,
    reg_lambda=0.05,
    random_state=42,
    force_col_wise=True
)

xgb_model = XGBRegressor(
    n_estimators=250,
    learning_rate=0.1,
    alpha=1.5,
    reg_lambda=0.015,
    random_state=42
)

# Define the range of features to test from 95 to 125
feature_range = np.arange(95, 126)  # Includes 125

# Setup for storing results
scores_lgbm = []
scores_xgb = []
kf = KFold(5)

# Loop to evaluate feature count
for n_features in feature_range:
    selector_lgbm = RFE(lgbm_model, n_features_to_select=n_features, step=3)
    selector_xgb = RFE(xgb_model, n_features_to_select=n_features, step=3)
    
    X_train_rfe_lgbm = selector_lgbm.fit_transform(X_train, y_train)
    X_train_rfe_xgb = selector_xgb.fit_transform(X_train, y_train)
    
    score_lgbm = cross_val_score(lgbm_model, X_train_rfe_lgbm, y_train, cv=kf, scoring='neg_mean_squared_error').mean()
    score_xgb = cross_val_score(xgb_model, X_train_rfe_xgb, y_train, cv=kf, scoring='neg_mean_squared_error').mean()
    
    scores_lgbm.append(score_lgbm)
    scores_xgb.append(score_xgb)

    # Save progress after each iteration
    with open('feature_selection_progress.pkl', 'wb') as f:
        pickle.dump((feature_range, scores_lgbm, scores_xgb), f)

# Load final scores (if needed)
with open('feature_selection_progress.pkl', 'rb') as f:
    feature_range, scores_lgbm, scores_xgb = pickle.load(f)

# Plotting results
plt.figure(figsize=(14, 7))

# LightGBM
plt.subplot(1, 2, 1)
plt.title("Feature Selection - LightGBM")
plt.xlabel("Number of features selected")
plt.ylabel("Cross-validation score (Neg MSE)")
plt.plot(feature_range, scores_lgbm, marker='o')
plt.grid(True)

# XGBoost
plt.subplot(1, 2, 2)
plt.title("Feature Selection - XGBoost")
plt.xlabel("Number of features selected")
plt.ylabel("Cross-validation score (Neg MSE)")
plt.plot(feature_range, scores_xgb, marker='o')
plt.grid(True)

plt.tight_layout()
plt.show()

# Output the optimal number of features
optimal_features_lgbm = feature_range[np.argmax(scores_lgbm)]
optimal_features_xgb = feature_range[np.argmax(scores_xgb)]

print("Optimal number of features for LightGBM:", optimal_features_lgbm)
print("Optimal number of features for XGBoost:", optimal_features_xgb)

In [ ]:
# Initialize models with optimal parameters
lgbm = LGBMRegressor(n_estimators=250, learning_rate=0.1, num_leaves=55, reg_alpha=1, reg_lambda=0.05, random_state=42)
xgb = XGBRegressor(n_estimators=250, learning_rate=0.1, alpha=1.5, reg_lambda=0.015, random_state=42)

# Apply RFE independently and train models directly on selected features
rfe_lgbm = RFE(estimator=lgbm, n_features_to_select=109, step=1)
rfe_xgb = RFE(estimator=xgb, n_features_to_select=103, step=1)

X_train_lgbm = rfe_lgbm.fit_transform(X_train, y_train)
X_test_lgbm = rfe_lgbm.transform(X_test)
X_train_xgb = rfe_xgb.fit_transform(X_train, y_train)
X_test_xgb = rfe_xgb.transform(X_test)

# Stacking the models correctly
meta_model = StackingRegressor(
    estimators=[('lgbm', lgbm), ('xgb', xgb)],
    final_estimator=LinearRegression(),
    passthrough=False  # Ensures only the predictions from lgbm and xgb are used as input to LinearRegression
)

# Train the meta model on the original dataset
meta_model.fit(np.hstack([X_train_lgbm, X_train_xgb]), y_train)
y_pred = meta_model.predict(np.hstack([X_test_lgbm, X_test_xgb]))

# Evaluation
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Meta Model Results:")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R-squared: {r2}")

In [ ]:
# Initialize models with optimal parameters
lgbm = LGBMRegressor(n_estimators=250, learning_rate=0.1, num_leaves=55, reg_alpha=1, reg_lambda=0.05, random_state=42)
xgb = XGBRegressor(n_estimators=250, learning_rate=0.1, alpha=1.5, reg_lambda=0.015, random_state=42)

# Apply RFE independently and train models directly on selected features
rfe_lgbm = RFE(estimator=lgbm, n_features_to_select=109, step=1)
rfe_xgb = RFE(estimator=xgb, n_features_to_select=103, step=1)

X_train_lgbm = rfe_lgbm.fit_transform(X_train, y_train)
X_test_lgbm = rfe_lgbm.transform(X_test)
X_train_xgb = rfe_xgb.fit_transform(X_train, y_train)
X_test_xgb = rfe_xgb.transform(X_test)

# Stacking the models correctly
meta_model = StackingRegressor(
    estimators=[('lgbm', lgbm), ('xgb', xgb)],
    final_estimator=LinearRegression(),
    passthrough=False  # Ensures only the predictions from lgbm and xgb are used as input to LinearRegression
)

# Train the meta model on the original dataset
meta_model.fit(np.hstack([X_train_lgbm, X_train_xgb]), y_train)
y_pred = meta_model.predict(np.hstack([X_test_lgbm, X_test_xgb]))

# Evaluation
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Meta Model Results:")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R-squared: {r2}")

In [ ]:
# Cross-validation for model stability
cv_scores = cross_val_score(meta_model, np.hstack([X_train_lgbm, X_train_xgb]), y_train, cv=KFold(10), scoring='r2')
print("10-fold Cross-validation R² scores:", cv_scores)
print("Mean R² across folds:", np.mean(cv_scores))

# Visualization of predicted vs actual values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, color='blue', alpha=0.6, label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=3, label='Ideal Fit')
plt.xlabel('Actual Output')
plt.ylabel('Predicted Output')
plt.title('Actual vs. Predicted Solar Power Output')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Initialize KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Concatenate training sets for cross-validation
X_combined_train = np.hstack([X_train_lgbm, X_train_xgb])
y_combined_train = y_train  # This is just for clarity; y_train is already your target array

# Compute cross-validation scores
cv_scores = cross_val_score(meta_model, X_combined_train, y_combined_train, cv=kf, scoring='r2')

# Plotting the cross-validation results
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), cv_scores, marker='o', linestyle='-', color='b')
plt.title('10-Fold Cross-Validation R^2 Scores')
plt.xlabel('Fold')
plt.ylabel('R^2 Score')
plt.xticks(np.arange(1, 11, step=1))  # Ensure ticks for each fold
plt.grid(True)
plt.show()

# Print the average and individual fold scores
print("Average R^2 Score:", np.mean(cv_scores))
print("R^2 Scores for each fold:", cv_scores)

In [ ]:
# Initialize KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Concatenate training sets for cross-validation
X_combined_train = np.hstack([X_train_lgbm, X_train_xgb])
y_combined_train = y_train  # This is just for clarity; y_train is already your target array

# Compute cross-validation scores
cv_scores = cross_val_score(meta_model, X_combined_train, y_combined_train, cv=kf, scoring='r2')

# Plotting the cross-validation results
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), cv_scores, marker='o', linestyle='-', color='b')
plt.title('10-Fold Cross-Validation R^2 Scores')
plt.xlabel('Fold')
plt.ylabel('R^2 Score')
plt.xticks(np.arange(1, 11, step=1))  # Ensure ticks for each fold
plt.grid(True)
plt.show()

# Print the average and individual fold scores
print("Average R^2 Score:", np.mean(cv_scores))
print("R^2 Scores for each fold:", cv_scores)

In [ ]:
# Assuming 'model' is your trained model and 'X_test', 'y_test' are your test datasets

# 1. Feature Importance Visualization for LightGBM and XGBoost
feature_importances_lgbm = pd.Series(lgbm.feature_importances_, index=X_train.columns[rfe_lgbm.support_])
feature_importances_xgb = pd.Series(xgb.feature_importances_, index=X_train.columns[rfe_xgb.support_])

plt.figure(figsize=(12, 6))
feature_importances_lgbm.sort_values().plot(kind='barh', title='Feature Importances in LightGBM')
plt.show()

plt.figure(figsize=(12, 6))
feature_importances_xgb.sort_values().plot(kind='barh', title='Feature Importances in XGBoost')
plt.show()

# 2. Residual Plot
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.title('Residual Plot')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.axhline(y=0, color='r', linestyle='--')
plt.grid(True)
plt.show()

# 3. Performance Metrics Across Different Conditions
# Example: Performance by different times of the day (assuming 'Hour' is a feature)
X_test['predicted'] = y_pred
X_test['actual'] = y_test
X_test['residual'] = residuals

# Group by hour and calculate RMSE
performance_by_hour = X_test.groupby('Hour').apply(lambda x: np.sqrt(mean_squared_error(x['actual'], x['predicted'])))
plt.figure(figsize=(10, 6))
performance_by_hour.plot(title='Model Performance by Hour of the Day')
plt.ylabel('RMSE')
plt.show()

# 4. Comparison with Baseline Models
# Assuming a simple linear regression as a baseline model
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train_lgbm, y_train)  # Fit on the same subset of features for a fair comparison
y_pred_lr = lr.predict(X_test_lgbm)

# Comparing R2 scores
r2_lr = r2_score(y_test, y_pred_lr)
print(f'R-squared for Linear Regression: {r2_lr}')
print(f'R-squared for Meta Model: {r2}')

# Visual comparison of predictions
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='blue', label='Meta Model Predictions')
plt.scatter(y_test, y_pred_lr, alpha=0.6, color='green', label='Linear Regression Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Output')
plt.ylabel('Predicted Output')
plt.title('Comparison of Meta Model and Linear Regression Predictions')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Assuming X_train, y_train are your features and target datasets

# 1. Feature Importance with Bayesian Ridge Regression
bayesian_ridge = BayesianRidge()
bayesian_ridge.fit(X_train, y_train)
feature_importance = pd.Series(np.abs(bayesian_ridge.coef_), index=X_train.columns)

# Plot feature importance
plt.figure(figsize=(10, 6))
feature_importance.nlargest(10).plot(kind='barh')
plt.title('Top 10 Important Features')
plt.show()

# 2. 10-fold Cross-Validation and Model Evaluation
kf = KFold(n_splits=10, shuffle=True, random_state=42)
mse_scores = cross_val_score(bayesian_ridge, X_train, y_train, scoring='neg_mean_squared_error', cv=kf)
r2_scores = cross_val_score(bayesian_ridge, X_train, y_train, scoring='r2', cv=kf)

print("10-fold CV Mean Squared Error:", -mse_scores.mean())
print("10-fold CV R-squared:", r2_scores.mean())

# 3. Visualize Predictions vs Actuals
y_pred = bayesian_ridge.predict(X_test)
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs. Predicted Power Output')
plt.show()

# 4. Detailed evaluation metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse}, RMSE: {rmse}, MAE: {mae}, R^2: {r2}")

In [ ]:
# Assuming X_train, y_train are your features and target datasets

# 1. Feature Importance with Bayesian Ridge Regression
bayesian_ridge = BayesianRidge()
bayesian_ridge.fit(X_train, y_train)
feature_importance = pd.Series(np.abs(bayesian_ridge.coef_), index=X_train.columns)

# Plot feature importance
plt.figure(figsize=(10, 6))
feature_importance.nlargest(10).plot(kind='barh')
plt.title('Top 10 Important Features')
plt.show()

# 2. 10-fold Cross-Validation and Model Evaluation
kf = KFold(n_splits=10, shuffle=True, random_state=42)
mse_scores = cross_val_score(bayesian_ridge, X_train, y_train, scoring='neg_mean_squared_error', cv=kf)
r2_scores = cross_val_score(bayesian_ridge, X_train, y_train, scoring='r2', cv=kf)

print("10-fold CV Mean Squared Error:", -mse_scores.mean())
print("10-fold CV R-squared:", r2_scores.mean())

# 3. Visualize Predictions vs Actuals
y_pred = bayesian_ridge.predict(X_test)
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs. Predicted Power Output')
plt.show()

# 4. Detailed evaluation metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse}, RMSE: {rmse}, MAE: {mae}, R^2: {r2}")

In [ ]:
# Re-initialize and refit the models on the selected features from RFE
# Ensure LightGBM uses the features selected from its own RFE
lgbm_model.fit(X_train_rfe_lgbm, y_train)

# Ensure XGBoost is refitted on the double-refined feature set determined by the second RFE
xgb_model.fit(X_train_rfe_xgb, y_train)

# Prepare features for the meta model using predictions from the base models as new inputs
# Use the predictions from both models as input features to the final meta model
stacked_features_train = np.column_stack([
    lgbm_model.predict(X_train_rfe_xgb),  # Ensure consistent feature usage
    xgb_model.predict(X_train_rfe_xgb)
])
stacked_features_test = np.column_stack([
    lgbm_model.predict(X_test_rfe_xgb),   # Ensure consistent feature usage
    xgb_model.predict(X_test_rfe_xgb)
])

# Define the meta-model; here we choose Linear Regression for simplicity and interpretability
meta_model = StackingRegressor(
    estimators=[
        ('lgb', lgbm_model),
        ('xgb', xgb_model)
    ],
    final_estimator=LinearRegression()
)

# Fit the meta-model on the stacked features
meta_model.fit(stacked_features_train, y_train)

# Predict using the meta-model on the test data
meta_y_pred = meta_model.predict(stacked_features_test)

# Evaluate the meta-model
meta_mse = mean_squared_error(y_test, meta_y_pred)
meta_rmse = np.sqrt(meta_mse)
meta_r2 = r2_score(y_test, meta_y_pred)

print("Meta-Model Stacking Results:")
print(f"MSE: {meta_mse}")
print(f"RMSE: {meta_rmse}")
print(f"R-squared: {meta_r2}")

In [ ]:
# Define the parameter distribution for XGBoost
param_dist_xgb = {
    'alpha': uniform(0.5, 1.5),  # Continuous distribution from 0.5 to 2
    'lambda': uniform(0.005, 0.02),  # Continuous distribution from 0.005 to 0.025
    'learning_rate': uniform(0.05, 0.2),  # Continuous distribution from 0.05 to 0.25
    'n_estimators': randint(100, 300)  # Discrete uniform distribution from 100 to 300
}

# Define the parameter distribution for LightGBM
param_dist_lgb = {
    'learning_rate': uniform(0.05, 0.2),  # Continuous distribution from 0.05 to 0.25
    'n_estimators': randint(100, 300),  # Discrete uniform distribution from 100 to 300
    'num_leaves': randint(30, 60),  # Discrete uniform distribution from 30 to 60
    'reg_alpha': uniform(0.5, 1.5),  # Continuous distribution from 0.5 to 2
    'reg_lambda': uniform(0.05, 0.2)  # Continuous distribution from 0.05 to 0.25
}

# Setup RandomizedSearchCV for XGBoost
random_search_xgb = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_distributions=param_dist_xgb,
    n_iter=100,  # Number of parameter settings that are sampled
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42
)

# Setup RandomizedSearchCV for LightGBM
random_search_lgb = RandomizedSearchCV(
    estimator=LGBMRegressor(random_state=42),
    param_distributions=param_dist_lgb,
    n_iter=100,  # Number of parameter settings that are sampled
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42
)

# Perform the random search on the reduced feature set
random_search_xgb.fit(X_train_rfe_xgb, y_train)
random_search_lgb.fit(X_train_rfe_lgb, y_train)

print("Random search best parameters for XGBoost:", random_search_xgb.best_params_)
print("Random search best score for XGBoost:", random_search_xgb.best_score_)
print("Random search best parameters for LightGBM:", random_search_lgb.best_params_)
print("Random search best score for LightGBM:", random_search_lgb.best_score_)

In [ ]:
# Define the parameter distribution for XGBoost
param_dist_xgb = {
    'alpha': uniform(0.5, 1.5),  # Continuous distribution from 0.5 to 2
    'lambda': uniform(0.005, 0.02),  # Continuous distribution from 0.005 to 0.025
    'learning_rate': uniform(0.05, 0.2),  # Continuous distribution from 0.05 to 0.25
    'n_estimators': randint(100, 300)  # Discrete uniform distribution from 100 to 300
}

# Define the parameter distribution for LightGBM
param_dist_lgb = {
    'learning_rate': uniform(0.05, 0.2),  # Continuous distribution from 0.05 to 0.25
    'n_estimators': randint(100, 300),  # Discrete uniform distribution from 100 to 300
    'num_leaves': randint(30, 60),  # Discrete uniform distribution from 30 to 60
    'reg_alpha': uniform(0.5, 1.5),  # Continuous distribution from 0.5 to 2
    'reg_lambda': uniform(0.05, 0.2)  # Continuous distribution from 0.05 to 0.25
}

# Setup RandomizedSearchCV for XGBoost
random_search_xgb = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_distributions=param_dist_xgb,
    n_iter=100,  # Number of parameter settings that are sampled
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42
)

# Setup RandomizedSearchCV for LightGBM
random_search_lgb = RandomizedSearchCV(
    estimator=LGBMRegressor(random_state=42),
    param_distributions=param_dist_lgb,
    n_iter=100,  # Number of parameter settings that are sampled
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42
)

# Perform the random search on the reduced feature set
random_search_xgb.fit(X_train_rfe_xgb, y_train)
random_search_lgb.fit(X_train_rfe_lgb, y_train)

print("Random search best parameters for XGBoost:", random_search_xgb.best_params_)
print("Random search best score for XGBoost:", random_search_xgb.best_score_)
print("Random search best parameters for LightGBM:", random_search_lgb.best_params_)
print("Random search best score for LightGBM:", random_search_lgb.best_score_)

In [ ]:
# Setup a more refined parameter grid around the best parameters
param_grid_xgb_refined = {
    'alpha': [0.5, 1, 1.5],
    'lambda': [0.005, 0.01, 0.015],
    'learning_rate': [0.05, 0.1, 0.15],
    'n_estimators': [150, 200, 250]
}

param_grid_lgb_refined = {
    'learning_rate': [0.05, 0.1, 0.15],
    'n_estimators': [150, 200, 250],
    'num_leaves': [45, 51, 55],
    'reg_alpha': [0.5, 1, 1.5],
    'reg_lambda': [0.05, 0.1, 0.15]
}

# Assume X_train_rfe and X_test_rfe are your feature-selected datasets
grid_search_xgb_refined = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=param_grid_xgb_refined,
    scoring='r2',
    cv=5,
    verbose=1
)

grid_search_lgb_refined = GridSearchCV(
    estimator=LGBMRegressor(random_state=42),
    param_grid=param_grid_lgb_refined,
    scoring='r2',
    cv=5,
    verbose=1
)

# Fit the grid search to the data
grid_search_xgb_refined.fit(X_train_rfe, y_train)
grid_search_lgb_refined.fit(X_train_rfe, y_train)

print("Refined best parameters for XGBoost:", grid_search_xgb_refined.best_params_)
print("Refined best score for XGBoost:", grid_search_xgb_refined.best_score_)
print("Refined best parameters for LightGBM:", grid_search_lgb_refined.best_params_)
print("Refined best score for LightGBM:", grid_search_lgb_refined.best_score_)

In [ ]:
# Re-initialize models with the best parameters found
optimized_lgb = LGBMRegressor(random_state=42, **grid_search_lgb_refined.best_params_)
optimized_xgb = XGBRegressor(random_state=42, **grid_search_xgb_refined.best_params_)

# Train the optimized models
optimized_lgb.fit(X_train_rfe, y_train)
optimized_xgb.fit(X_train_rfe, y_train)

# Stack the optimized models
optimized_stacked_features_train = np.hstack([
    optimized_lgb.predict(X_train_rfe).reshape(-1, 1),
    optimized_xgb.predict(X_train_rfe).reshape(-1, 1)
])
optimized_stacked_features_test = np.hstack([
    optimized_lgb.predict(X_test_rfe).reshape(-1, 1),
    optimized_xgb.predict(X_test_rfe).reshape(-1, 1)
])

# Fit the final stacking regressor with optimized models
optimized_stack_rfe = StackingRegressor(
    estimators=[('lgb', optimized_lgb), ('xgb', optimized_xgb)],
    final_estimator=LinearRegression()
)
optimized_stack_rfe.fit(optimized_stacked_features_train, y_train)

# Predict and evaluate using the optimized stacked model
optimized_y_pred = optimized_stack_rfe.predict(optimized_stacked_features_test)
optimized_mse = mean_squared_error(y_test, optimized_y_pred)
optimized_rmse = np.sqrt(optimized_mse)
optimized_mae = mean_absolute_error(y_test, optimized_y_pred)
optimized_r2 = r2_score(y_test, optimized_y_pred)

print(f"Optimized Stacking Model MSE: {optimized_mse}")
print(f"Optimized Stacking Model R-squared: {optimized_r2}")
print(f"Optimized Stacking Model MAE: {optimized_mae}")
print(f"Optimized Stacking Model RMSE: {optimized_rmse}")

In [ ]:
# Define the range of features to test
feature_counts = range(1, 15, 1)  # Testing from 1 to 50 features, in steps of 5
performance_scores = []

for n_features in feature_counts:
    # RFE with LightGBM as an example
    lgb_rfe = RFE(estimator=LGBMRegressor(random_state=42), n_features_to_select=n_features)
    lgb_rfe.fit(X_train, y_train)
    X_train_rfe = X_train.loc[:, lgb_rfe.support_]

    # Evaluate model with cross-validation
    model = LGBMRegressor(random_state=42)
    scores = cross_val_score(model, X_train_rfe, y_train, scoring='r2', cv=5)
    mean_score = np.mean(scores)
    performance_scores.append(mean_score)
    print(f"Tested {n_features} features: R-squared = {mean_score}")

# Identify the number of features that led to the best average R-squared
optimal_features = feature_counts[np.argmax(performance_scores)]
print(f"Optimal number of features: {optimal_features} with R-squared: {max(performance_scores)}")
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(feature_counts, performance_scores, marker='o')
plt.title('Model Performance vs. Number of Features')
plt.xlabel('Number of Features')
plt.ylabel('Cross-Validated R-squared')
plt.grid(True)
plt.show()

In [ ]:
# Recursive Feature Elimination (RFE) with LightGBM and XGBoost
from sklearn.feature_selection import RFE
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
# LightGBM RFE
lgb_rfe = RFE(estimator=LGBMRegressor(random_state=42), n_features_to_select=39)
lgb_rfe.fit(X, y)
X_rfe_lgb = X.loc[:, lgb_rfe.support_]
# XGBoost RFE
xgb_rfe = RFE(estimator=XGBRegressor(random_state=42), n_features_to_select=39)
xgb_rfe.fit(X, y)
X_rfe_xgb = X.loc[:, xgb_rfe.support_]

In [ ]:
#  SHAP values for feature importance with LightGBM and XGBoost


# LightGBM SHAP
model_lgb = LGBMRegressor(random_state=42)
model_lgb.fit(X, y)
explainer_lgb = shap.Explainer(model_lgb)
shap_values_lgb = explainer_lgb.shap_values(X)
shap.summary_plot(shap_values_lgb, X, plot_type="bar")

# Select top features based on SHAP values
shap_importance_lgb = np.abs(shap_values_lgb).mean(axis=0)
shap_top_features_lgb = X.columns[np.argsort(shap_importance_lgb)[-10:]]
X_shap_lgb = X[shap_top_features_lgb]

# XGBoost SHAP
model_xgb = XGBRegressor(random_state=42)
model_xgb.fit(X, y)
explainer_xgb = shap.Explainer(model_xgb)
shap_values_xgb = explainer_xgb.shap_values(X)
shap.summary_plot(shap_values_xgb, X, plot_type="bar")

# Select top features based on SHAP values
shap_importance_xgb = np.abs(shap_values_xgb).mean(axis=0)
shap_top_features_xgb = X.columns[np.argsort(shap_importance_xgb)[-10:]]
X_shap_xgb = X[shap_top_features_xgb]

In [ ]:
#  Model Training with LightGBM and XGBoost using RFE features
lgb_rfe_model = LGBMRegressor(random_state=42)
lgb_rfe_model.fit(X_rfe_lgb, y)
xgb_rfe_model = XGBRegressor(random_state=42)
xgb_rfe_model.fit(X_rfe_xgb, y)
# Hyperparameter Tuning for LightGBM and XGBoost
from sklearn.model_selection import RandomizedSearchCV

# LightGBM hyperparameter tuning
lgb_param_grid = {
    'learning_rate': [0.01, 0.1, 0.05],
    'n_estimators': [100, 200, 300],
    'num_leaves': [31, 50, 70]
}
lgb_search = RandomizedSearchCV(LGBMRegressor(random_state=42), param_distributions=lgb_param_grid, n_iter=50, cv=5, scoring='neg_mean_squared_error')
lgb_search.fit(X, y)
print(f"Best parameters for LightGBM: {lgb_search.best_params_}")

# XGBoost hyperparameter tuning
xgb_param_grid = {
    'learning_rate': [0.01, 0.1, 0.05],
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 9]
}
xgb_search = RandomizedSearchCV(XGBRegressor(random_state=42), param_distributions=xgb_param_grid, n_iter=50, cv=5, scoring='neg_mean_squared_error')
xgb_search.fit(X, y)
print(f"Best parameters for XGBoost: {xgb_search.best_params_}")
#  Advanced Stacking Model with RFE features
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Stacking with RFE features
stack_rfe = StackingRegressor(
    estimators=[('lgb', lgb_rfe_model), ('xgb', xgb_rfe_model)],
    final_estimator=LinearRegression()
)
stack_rfe.fit(np.hstack([lgb_rfe_model.predict(X_rfe_lgb).reshape(-1, 1), xgb_rfe_model.predict(X_rfe_xgb).reshape(-1, 1)]), y)
mae = mean_absolute_error(y, stack_rfe.predict(np.hstack([lgb_rfe_model.predict(X_rfe_lgb).reshape(-1, 1), xgb_rfe_model.predict(X_rfe_xgb).reshape(-1, 1)])))
rmse = mean_squared_error(y, stack_rfe.predict(np.hstack([lgb_rfe_model.predict(X_rfe_lgb).reshape(-1, 1), xgb_rfe_model.predict(X_rfe_xgb).reshape(-1, 1)])), squared=False)
# Evaluation
print(f"Stacking Model with RFE features MSE: {mean_squared_error(y, stack_rfe.predict(np.hstack([lgb_rfe_model.predict(X_rfe_lgb).reshape(-1, 1), xgb_rfe_model.predict(X_rfe_xgb).reshape(-1, 1)])))}")
print(f"Stacking Model with RFE features R-squared: {stack_rfe.score(np.hstack([lgb_rfe_model.predict(X_rfe_lgb).reshape(-1, 1), xgb_rfe_model.predict(X_rfe_xgb).reshape(-1, 1)]), y)}")
print(f"Stacking Model with RFE features MAE: {mae}")
print(f"Stacking Model with RFE features RMSE: {rmse}")

In [ ]:
# Defining the parameter grid for XGBoost and LightGBM
param_grid_xgb = {
    'alpha': [0.001, 0.01, 0.1, 1],
    'lambda': [0.001, 0.01, 0.1, 1]
}

param_grid_lgb = {
    'lambda_l1': [0.001, 0.01, 0.1, 1],
    'lambda_l2': [0.001, 0.01, 0.1, 1]
}

# Setting up the GridSearchCV for XGBoost
grid_search_xgb = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=param_grid_xgb,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=1
)

# Setting up the GridSearchCV for LightGBM
grid_search_lgb = GridSearchCV(
    estimator=LGBMRegressor(random_state=42),
    param_grid=param_grid_lgb,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=1
)

# Assuming X_rfe_xgb and X_rfe_lgb are the datasets with features selected by RFE
# Fit the grid search to the data
grid_search_xgb.fit(X_rfe_xgb, y)
grid_search_lgb.fit(X_rfe_lgb, y)

# Print the best parameters and best score
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)
print("Best score for XGBoost:", grid_search_xgb.best_score_)
print("Best parameters for LightGBM:", grid_search_lgb.best_params_)
print("Best score for LightGBM:", grid_search_lgb.best_score_)

In [ ]:
# Defining the parameter grid for XGBoost and LightGBM
param_grid_xgb = {
    'alpha': [0.001, 0.01, 0.1, 1],
    'lambda': [0.001, 0.01, 0.1, 1]
}

param_grid_lgb = {
    'lambda_l1': [0.001, 0.01, 0.1, 1],
    'lambda_l2': [0.001, 0.01, 0.1, 1]
}

# Setting up the GridSearchCV for XGBoost
grid_search_xgb = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=param_grid_xgb,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=1
)

# Setting up the GridSearchCV for LightGBM
grid_search_lgb = GridSearchCV(
    estimator=LGBMRegressor(random_state=42),
    param_grid=param_grid_lgb,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=1
)

# Assuming X_rfe_xgb and X_rfe_lgb are the datasets with features selected by RFE
# Fit the grid search to the data
grid_search_xgb.fit(X_rfe_xgb, y)
grid_search_lgb.fit(X_rfe_lgb, y)

# Print the best parameters and best score
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)
print("Best score for XGBoost:", grid_search_xgb.best_score_)
print("Best parameters for LightGBM:", grid_search_lgb.best_params_)
print("Best score for LightGBM:", grid_search_lgb.best_score_)
# Correcting the parameter names and setting up the optimized models
xgb_optimized = XGBRegressor(random_state=42, alpha=grid_search_xgb.best_params_['alpha'], reg_lambda=grid_search_xgb.best_params_['lambda'])
lgb_optimized = LGBMRegressor(random_state=42, lambda_l1=grid_search_lgb.best_params_['lambda_l1'], lambda_l2=grid_search_lgb.best_params_['lambda_l2'])

# Fit the optimized models on RFE selected features
xgb_optimized.fit(X_rfe_xgb, y)
lgb_optimized.fit(X_rfe_lgb, y)

# Preparing features for the stacking model
stacked_features_train = np.hstack([xgb_optimized.predict(X_rfe_xgb).reshape(-1, 1), lgb_optimized.predict(X_rfe_lgb).reshape(-1, 1)])
stacked_features_test = np.hstack([xgb_optimized.predict(X_test_rfe_xgb).reshape(-1, 1), lgb_optimized.predict(X_test_rfe_lgb).reshape(-1, 1)])

# Stacking models
stack_rfe = StackingRegressor(
    estimators=[('xgb', xgb_optimized), ('lgb', lgb_optimized)],
    final_estimator=LinearRegression()
)

# Fit the stacking model
stack_rfe.fit(stacked_features_train, y_train)

# Predict using the stacking model
y_pred = stack_rfe.predict(stacked_features_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Stacking Model with RFE features MSE: {mse}")
print(f"Stacking Model with RFE features R-squared: {r2}")
print(f"Stacking Model with RFE features MAE: {mae}")
print(f"Stacking Model with RFE features RMSE: {rmse}")

In [ ]:
# Assuming X and y are already defined and preprocessed
kf = KFold(n_splits=10, shuffle=True, random_state=42)

mse_scores = []
rmse_scores = []
mae_scores = []
r2_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Fit RFE models and select features
    lgb_rfe = RFE(estimator=LGBMRegressor(random_state=42), n_features_to_select=39)
    xgb_rfe = RFE(estimator=XGBRegressor(random_state=42), n_features_to_select=39)

    lgb_rfe.fit(X_train, y_train)
    xgb_rfe.fit(X_train, y_train)

    X_train_rfe_lgb = X_train.loc[:, lgb_rfe.support_]
    X_test_rfe_lgb = X_test.loc[:, lgb_rfe.support_]
    X_train_rfe_xgb = X_train.loc[:, xgb_rfe.support_]
    X_test_rfe_xgb = X_test.loc[:, xgb_rfe.support_]

    # Train models on RFE selected features
    lgb_model = LGBMRegressor(random_state=42)
    xgb_model = XGBRegressor(random_state=42)
    
    lgb_model.fit(X_train_rfe_lgb, y_train)
    xgb_model.fit(X_train_rfe_xgb, y_train)

    # Stack models
    stack_rfe = StackingRegressor(
        estimators=[('lgb', lgb_model), ('xgb', xgb_model)],
        final_estimator=LinearRegression()
    )

    # Prepare stacked features
    stacked_features_train = np.hstack([lgb_model.predict(X_train_rfe_lgb).reshape(-1, 1), 
                                        xgb_model.predict(X_train_rfe_xgb).reshape(-1, 1)])
    stacked_features_test = np.hstack([lgb_model.predict(X_test_rfe_lgb).reshape(-1, 1), 
                                       xgb_model.predict(X_test_rfe_xgb).reshape(-1, 1)])

    stack_rfe.fit(stacked_features_train, y_train)
    y_pred = stack_rfe.predict(stacked_features_test)

    # Collect metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mse_scores.append(mse)
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

# Display average results from 10-fold CV
print("Average MSE:", np.mean(mse_scores))
print("Average RMSE:", np.mean(rmse_scores))
print("Average MAE:", np.mean(mae_scores))
print("Average R²:", np.mean(r2_scores))

In [ ]:
# Advanced Stacking Model with SHAP features
# Stacking with SHAP features
model_lgb.fit(X_shap_lgb, y)
model_xgb.fit(X_shap_xgb, y)
stack_shap = StackingRegressor(
    estimators=[('lgb', model_lgb), ('xgb', model_xgb)],
    final_estimator=LinearRegression()
)
stack_shap.fit(np.hstack([model_lgb.predict(X_shap_lgb).reshape(-1, 1), model_xgb.predict(X_shap_xgb).reshape(-1, 1)]), y)

# Evaluation
print(f"Stacking Model with SHAP features MSE: {mean_squared_error(y, stack_shap.predict(np.hstack([model_lgb.predict(X_shap_lgb).reshape(-1, 1), model_xgb.predict(X_shap_xgb).reshape(-1, 1)])))}")
print(f"Stacking Model with SHAP features R-squared: {stack_shap.score(np.hstack([model_lgb.predict(X_shap_lgb).reshape(-1, 1), model_xgb.predict(X_shap_xgb).reshape(-1, 1)]), y)}")

# Cross-Validation Scores with SHAP features for both models
from sklearn.model_selection import cross_val_score

cv_scores_shap = cross_val_score(stack_shap, np.hstack([model_lgb.predict(X_shap_lgb).reshape(-1, 1), model_xgb.predict(X_shap_xgb).reshape(-1, 1)]), y, cv=5, scoring='neg_mean_squared_error')
print(f"Stacking Model with SHAP features Cross-Validation MSE: {-cv_scores_shap.mean()} ± {cv_scores_shap.std()}")

# Cross-Validation Scores with RFE features for both models
cv_scores_rfe = cross_val_score(stack_rfe, np.hstack([lgb_rfe_model.predict(X_rfe_lgb).reshape(-1, 1), xgb_rfe_model.predict(X_rfe_xgb).reshape(-1, 1)]), y, cv=5, scoring='neg_mean_squared_error')
print(f"Stacking Model with RFE features Cross-Validation MSE: {-cv_scores_rfe.mean()} ± {cv_scores_rfe.std()}")
#  Bayesian Optimization for Hyperparameters
from skopt import BayesSearchCV

# Perform Bayesian Optimization for LightGBM
lgb_bayes_params = {
    'learning_rate': (0.01, 0.1, 'log-uniform'),
    'n_estimators': (100, 500),
    'num_leaves': (31, 100)
}
lgb_bayes_search = BayesSearchCV(LGBMRegressor(random_state=42), lgb_bayes_params, n_iter=50, cv=5, scoring='neg_mean_squared_error')
lgb_bayes_search.fit(X, y)
print(f"Best parameters for LightGBM: {lgb_bayes_search.best_params_}")

# Perform Bayesian Optimization for XGBoost
xgb_bayes_params = {
    'learning_rate': (0.01, 0.1, 'log-uniform'),
    'n_estimators': (100, 500),
    'max_depth': (3, 10)
}
xgb_bayes_search = BayesSearchCV(XGBRegressor(random_state=42), xgb_bayes_params, n_iter=50, cv=5, scoring='neg_mean_squared_error')
xgb_bayes_search.fit(X, y)
print(f"Best parameters for XGBoost: {xgb_bayes_search.best_params_}")

In [ ]:
# Final Stacking Model with Bayesian Optimized Base Learners
# Define the base models with optimized hyperparameters
lgb_bayes_optimized = LGBMRegressor(**lgb_bayes_search.best_params_)
xgb_bayes_optimized = XGBRegressor(**xgb_bayes_search.best_params_)

# Train the base models
lgb_bayes_optimized.fit(X, y)
xgb_bayes_optimized.fit(X, y)

# Define the stacking model with Bayesian optimized base learners
stack_bayes = StackingRegressor(
    estimators=[('lgb', lgb_bayes_optimized), ('xgb', xgb_bayes_optimized)],
    final_estimator=LinearRegression()
)
stack_bayes.fit(np.hstack([lgb_bayes_optimized.predict(X).reshape(-1, 1), xgb_bayes_optimized.predict(X).reshape(-1, 1)]), y)

# Evaluate the stacking model
print(f"Stacking Model with Bayesian Optimized Base Learners MSE: {mean_squared_error(y, stack_bayes.predict(np.hstack([lgb_bayes_optimized.predict(X).reshape(-1, 1), xgb_bayes_optimized.predict(X).reshape(-1, 1)])))}")
print(f"Stacking Model with Bayesian Optimized Base Learners R-squared: {stack_bayes.score(np.hstack([lgb_bayes_optimized.predict(X).reshape(-1, 1), xgb_bayes_optimized.predict(X).reshape(-1, 1)]), y)}")

# Cross-validation for the Bayesian optimized models
cv_scores_bayes_lgb = cross_val_score(stack_bayes, np.hstack([lgb_bayes_optimized.predict(X).reshape(-1, 1), xgb_bayes_optimized.predict(X).reshape(-1, 1)]), y, cv=5, scoring='neg_mean_squared_error')
print(f"Bayesian Optimized LightGBM Cross-Validation MSE: {-cv_scores_bayes_lgb.mean()} ± {cv_scores_bayes_lgb.std()}")

cv_scores_bayes_xgb = cross_val_score(stack_bayes, np.hstack([lgb_bayes_optimized.predict(X).reshape(-1, 1), xgb_bayes_optimized.predict(X).reshape(-1, 1)]), y, cv=5, scoring='neg_mean_squared_error')
print(f"Bayesian Optimized XGBoost Cross-Validation MSE: {-cv_scores_bayes_xgb.mean()} ± {cv_scores_bayes_xgb.std()}")
from sklearn.model_selection import learning_curve

train_sizes, train_scores, validation_scores = learning_curve(
    estimator = stack_rfe,
    X = X, y = y,
    train_sizes = [0.1, 0.33, 0.55, 0.78, 1.],
    cv = 5,
    scoring = 'neg_mean_squared_error'
)

train_scores_mean = -train_scores.mean(axis=1)
validation_scores_mean = -validation_scores.mean(axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
plt.plot(train_sizes, validation_scores_mean, 'o-', color="g", label="Cross-validation score")
plt.title('Learning curves')
plt.xlabel('Training examples')
plt.ylabel('MSE')
plt.legend(loc="best")
plt.grid(True)
plt.show()
# Assuming X and y are your full dataset and target variable

# Apply the trained RFE and model to get predictions
X_rfe_lgb = lgb_rfe.transform(X)
X_rfe_xgb = xgb_rfe.transform(X)
predictions = stack_rfe.predict(np.hstack([lgb_rfe_model.predict(X_rfe_lgb).reshape(-1, 1), xgb_rfe_model.predict(X_rfe_xgb).reshape(-1, 1)]))

# Plotting Predicted vs Actual
plt.figure(figsize=(10, 6))
plt.scatter(y, predictions, alpha=0.3, color='blue')  # Alpha for transparency in the dots
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=4)  # Line for perfect predictions
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs. Actual Plot')
import numpy as np
import matplotlib.pyplot as plt

# Apply the trained RFE and model to get predictions
X_rfe_lgb = lgb_rfe.transform(X)
X_rfe_xgb = xgb_rfe.transform(X)
predictions = stack_rfe.predict(np.hstack([lgb_rfe_model.predict(X_rfe_lgb).reshape(-1, 1), xgb_rfe_model.predict(X_rfe_xgb).reshape(-1, 1)]))

# Plotting the residual histogram
plt.figure(figsize=(10, 6))
plt.hist(residuals, bins=30, color='blue', alpha=0.7)
plt.title('Histogram of Residuals')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
# Assuming X is the original DataFrame and you have already used RFE for feature selection
# Ensure X is a DataFrame with column names

# Extract names of the features selected by RFE for both LightGBM and XGBoost
lgb_feature_names = X.columns[lgb_rfe.support_]
xgb_feature_names = X.columns[xgb_rfe.support_]

# LightGBM feature importances
lgb_importances = lgb_rfe_model.feature_importances_
lgb_indices = np.argsort(lgb_importances)[-10:]  # Get indices of top 15 features

# XGBoost feature importances
xgb_booster = xgb_rfe_model.get_booster()
xgb_importances = xgb_booster.get_score(importance_type='weight')
# Sort the importances and extract the top 15
xgb_top_features = sorted(xgb_importances, key=xgb_importances.get, reverse=True)[:10]
xgb_top_importances = [xgb_importances[feat] for feat in xgb_top_features]

# Plot LightGBM Feature Importances
plt.figure(figsize=(12, 8))
plt.title('Top 10 Feature Importances in LightGBM Model')
plt.barh(range(10), lgb_importances[lgb_indices], color='b', align='center')
plt.yticks(range(10), [lgb_feature_names[i] for i in lgb_indices])
plt.xlabel('Relative Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important feature at the top
plt.show()

# Plot XGBoost Feature Importances
plt.figure(figsize=(12, 8))
plt.title('Top 10 Feature Importances in XGBoost Model')
plt.barh(range(10), xgb_top_importances, color='r', align='center')
plt.yticks(range(10), xgb_top_features)
plt.xlabel('Relative Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important feature at the top
plt.show()

In [ ]:
mse_scores = []
feature_range = range(24, 45, 1)  # Testing with step size of 5

for n_features in feature_range:
    print(f"Testing {n_features} features...")
    # Initialize and fit RFE with increased step size for faster reduction
    selector = RFE(estimator=lgb_rfe_model, n_features_to_select=n_features, step=1)
    X_rfe = selector.fit_transform(X, y)

    # Define the stacking model using the selected features
    stack_rfe = StackingRegressor(
        estimators=[('lgb', lgb_rfe_model), ('xgb', xgb_rfe_model)],
        final_estimator=LinearRegression()
    )

    # Evaluate the model using cross-validation
    mse = -np.mean(cross_val_score(stack_rfe, X_rfe, y, cv=5, scoring='neg_mean_squared_error'))
    mse_scores.append(mse)

    # Cleanup to free memory
    del selector, X_rfe, stack_rfe
    gc.collect()

# Plotting the MSE results
plt.figure(figsize=(10, 6))
plt.plot(feature_range, mse_scores, marker='o', linestyle='-', color='b')
plt.title('Cross-Validation MSE for Stacking Model with Varying Number of RFE Features')
plt.xlabel('Number of Features')
plt.ylabel('Negative MSE')
plt.grid(True)
plt.show()
# Save the figure
plt.savefig('plot.png') 

In [ ]:
# Assuming y_test are your actual values and y_pred are predictions from your model
# y_test and y_pred need to be of the same length and previously defined in your workflow

# Generate some example data if you haven't already
np.random.seed(0)
y_test = np.random.rand(4000) * 2  # Random values multiplied to scale similar to your plot
y_pred = y_test + np.random.normal(0, 0.1, 4000)  # Adding random noise to actual values

plt.figure(figsize=(10, 6))
plt.plot(y_test, label='Actual Values', color='red')
plt.plot(y_pred, label='Predicted Values', color='blue', alpha=0.5)  # Alpha for transparency
plt.title('Comparison of Actual and Predicted Values by Meta Model')
plt.xlabel('Samples')
plt.ylabel('PolyPV Output')
plt.legend()
plt.show()
# Save the figure
plt.savefig('plot.png') 

In [ ]:
import os
print(os.getcwd())